# Mistério em João Pessoa

Um assassinato foi registrado em João Pessoa. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar — sem nenhum pipeline pronto por trás, dessa vez é você quem constrói.

**Ferramentas: só pandas + MinIO + Jupyter.** Tudo se resolve com pandas puro e 2 funções de `utils`: `write_table(...)` e `read_table(...)` (publicar e ler uma tabela). Ilustradas com exemplo logo na Fase 1, abaixo. Filtros, joins e agregações são sempre pandas puro (`merge`, `groupby`, filtro booleano...).

## O que você recebeu

6 CSVs em `dados/`:

| Arquivo | O que é |
|---|---|
| `ocorrencia.csv` | O registro da ocorrência (data, tipo, cidade, descrição) |
| `pessoa.csv` | Cadastro de pessoas (nome, endereço) |
| `cnh.csv` | Dados de CNH — quem tem carteira de motorista, placa e veículo |
| `depoimento.csv` | Depoimentos de testemunhas |
| `membro_academia.csv` | Matrículas de uma academia da cidade |
| `checkin_academia.csv` | Check-ins de entrada na academia |

Sim, faltou combinar os formatos de data entre as tabelas — cada uma pode vir de um jeito diferente (texto, ISO, número). É de propósito: parte do trabalho de qualquer engenheiro de dados é notar isso e não deixar passar batido.

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`write_table(df, "bronze", "<nome>")`).
2. **Fase 2 — Investigação**: livre — use pandas (`read_table` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

Sem spoiler aqui — a história e as pistas estão só nos dados. Boa investigação!

In [9]:
from utils import list_layer, read_table, write_table
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos:

```python
df = pd.read_csv("dados/<arquivo>.csv")                # lê o CSV cru direto do disco
write_table(df, "bronze", "<nome_tabela>")            # publica: grava Parquet
```

Feito isso, `list_layer("bronze")` (ou o MinIO Console, http://localhost:9001) mostra os arquivos aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida das funções auxiliares de leitura logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [10]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
write_table(df_ocorrencia, "bronze", "ocorrencia")
df_ocorrencia

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,"Corpo encontrado nas imediações da academia Letsvibe Quadramares 24h, bairro de Quadramares, por volta das 09h10. Testemunha 1 mora na casa de numero_endereco mais alto da Avenida Ministro José Américo de Almeida. Testemunha 2 se chama Cida (apelido de Maria Aparecida) e mora na Avenida Rui Carneiro.",João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizinha.,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oceania.,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


### Funções auxiliares de leitura (exemplo rápido)

Duas funções que você vai usar sem parar a partir daqui — servem só para **ler**, ilustradas com a tabela que acabamos de publicar:

- `list_layer(layer)` lista os arquivos físicos de uma camada (sem ler o conteúdo) — útil pra conferir o que existe sem precisar abrir o MinIO Console.
- `read_table(layer, tabela)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

In [4]:
# O que já existe fisicamente na camada bronze (só "ocorrencia" até agora)
list_layer("bronze")

['bronze/.keep', 'bronze/ocorrencia/ocorrencia.parquet']

In [5]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
read_table("bronze", "ocorrencia")

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,Corpo encontrado nas imediações da academia Le...,João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizi...,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oce...,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


In [ ]:
# TODO: repita o padrão da célula do exemplo para "pessoa.csv" -> bronze.pessoa


In [ ]:
# TODO: repita o padrão para "cnh.csv" -> bronze.cnh


In [ ]:
# TODO: repita o padrão para "depoimento.csv" -> bronze.depoimento


In [ ]:
# TODO: repita o padrão para "membro_academia.csv" -> bronze.membro_academia


In [ ]:
# TODO: repita o padrão para "checkin_academia.csv" -> bronze.checkin_academia


In [6]:
# Checagem: as 6 tabelas devem aparecer aqui
list_layer("bronze")

['bronze/.keep', 'bronze/ocorrencia/ocorrencia.parquet']

## Fase 2 — Investigação (pandas)

A partir de aqui é livre: leia as tabelas bronze direto do MinIO com `read_table("bronze", "<tabela>")` (pandas lendo o Parquet), e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Ache a ocorrência (tipo = assassinato, cidade = João Pessoa) e leia a `descricao` com atenção — ela dá pistas de **endereço** para achar testemunhas em `pessoa`.
2. Ache as testemunhas em `pessoa` e cruze com `depoimento` para ler o que cada uma contou.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` (algo sobre o plano/data de matrícula), a outra para `cnh` (algo sobre o veículo/placa).
4. Filtre cada tabela pela pista correspondente. Sozinha, cada pista pode sobrar **mais de 1** candidato — o cruzamento das duas é que deve fechar em exatamente **1** pessoa.
5. (Bônus) Confirme em `checkin_academia` que o suspeito realmente esteve na academia por perto do horário da ocorrência.

Lembre-se: as datas não vêm todas no mesmo formato entre as tabelas — repare em cada uma antes de comparar/filtrar por data.

In [8]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = read_table("bronze", "ocorrencia")
# pessoa = read_table("bronze", "pessoa")
# cnh = read_table("bronze", "cnh")
# depoimento = read_table("bronze", "depoimento")
# membro_academia = read_table("bronze", "membro_academia")
# checkin_academia = read_table("bronze", "checkin_academia")

ocorrencia

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,Corpo encontrado nas imediações da academia Le...,João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizi...,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oce...,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


### Passo 1 — a ocorrência

Filtre `ocorrencia` para achar o assassinato em João Pessoa, e leia a `descricao` inteira (`print(...)` ajuda a não truncar o texto).

In [ ]:
# TODO: filtre ocorrencia por tipo == "assassinato" e cidade == "João Pessoa"
# e dê print() na coluna "descricao" da linha encontrada.


### Passo 2 — as testemunhas

A descrição da ocorrência aponta para 2 pessoas em `pessoa`, cada uma identificada por uma pista de **rua/endereço** diferente (uma pelo número mais alto numa rua, a outra pelo nome numa outra rua). Ache as duas, depois cruze os `id` delas com `depoimento.pessoa_id` para ler o que cada uma contou.

In [ ]:
# TODO: ache as 2 testemunhas em `pessoa` (via as pistas de rua/endereço da descrição)
# e depois os depoimentos delas em `depoimento` (merge por pessoa_id, ou filtro direto).


### Passo 3 — duas pistas, duas tabelas

Um depoimento descreve algo sobre o **plano e a data de matrícula** de alguém na academia — filtre `membro_academia` por isso. Sozinha, essa pista deve deixar **mais de 1** candidato (tudo bem, é assim mesmo).

O outro depoimento descreve algo sobre o **veículo/placa** de quem fugiu — filtre `cnh` por isso.

In [ ]:
# TODO: filtre `membro_academia` pela pista do 1o depoimento (plano + data de matrícula)
# candidatos_academia = ...


In [ ]:
# TODO: filtre `cnh` pela pista do 2o depoimento (veículo/placa)
# candidatos_placa = ...


### Passo 4 — cruzando as pistas

Cruze `candidatos_academia` com `candidatos_placa` (por `pessoa_id`/`id`) — deve sobrar exatamente **1** pessoa. Se sobrar mais de 1 (ou 0), revise os filtros dos passos anteriores.

In [ ]:
# TODO: cruze os dois conjuntos de candidatos e confira que sobrou 1 só suspeito
# suspeito = ...


### Passo 5 (bônus) — confirmar com o check-in

Cruze `membro_academia` com `checkin_academia` (por `matricula_id`) e confira que o suspeito tem um check-in na data da ocorrência, num horário compatível com o depoimento. Repare no formato da data em `checkin_academia` — é diferente do formato usado em `ocorrencia`.

In [ ]:
# TODO (bônus): confirme o check-in do suspeito na data/horário da ocorrência


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `cnh`) que fechou o caso |
| `pista_academia` | qual plano/data de matrícula bateu com o depoimento |
| `pista_veiculo` | qual detalhe do veículo bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `write_table(df_resposta, "silver", "resposta_caso")` — depois disso, `read_table("silver", "resposta_caso")` (ou o MinIO Console, em `lakehouse/silver/resposta_caso/`) já mostra o resultado.

In [ ]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver
# df_resposta = pd.DataFrame({...})
# write_table(df_resposta, "silver", "resposta_caso")
# read_table("silver", "resposta_caso")


---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso/` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.